# 01 — Wrangle & Feature Engineer

**Purpose:** Load validated transactions, add time/classification features,
build monthly aggregates per category.

**Inputs:** `data/interim/transactions_validated.parquet`  
**Outputs:**
- `data/interim/transactions_featured.parquet`
- `data/interim/monthly_aggregates.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from src.utils.logging_config import setup_notebook_logging
from src.analysis.feature_engineering import (
    engineer_features,
    build_monthly_aggregates,
    build_monthly_cashflow_split,
)
from src.storage.local_writer import LocalWriter

logger = setup_notebook_logging()
writer = LocalWriter(project_root="..")


In [ ]:
# ── Load validated transactions ───────────────────────────────────────────────
df = writer.load_interim("transactions_validated")
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# ── Engineer features ─────────────────────────────────────────────────────────
df_featured = engineer_features(df)
print(f"\nNew columns added: {[c for c in df_featured.columns if c not in df.columns]}")
display(df_featured.head())

In [ ]:
# ── Transfers summary ─────────────────────────────────────────────────────────
n_transfers = df_featured["is_transfer"].sum()
print(f"Transfers (excluded from category analysis): {n_transfers} ({n_transfers/len(df_featured):.1%})")

In [ ]:
# ── Monthly aggregates ────────────────────────────────────────────────────────
monthly = build_monthly_aggregates(df_featured)
print(f"\nMonthly aggregates: {monthly.shape[0]} rows")
display(monthly.head(20))

In [ ]:
monthly['year_month'].unique()

In [ ]:
# ── Top categories by total spend (absolute value) ────────────────────────────
top_cats = (
    monthly.groupby("category")["total_amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
    .head(15)
)
display(top_cats.rename("absolute_spend").to_frame())

In [ ]:
# ── Net cash flow per month (income - expenses, excluding transfers) ───────────
net_monthly = (
    monthly.groupby("year_month")["total_amount"]
    .sum()
    .reset_index()
    .rename(columns={"total_amount": "net_cashflow"})
)

colors = ["steelblue" if v >= 0 else "tomato" for v in net_monthly["net_cashflow"]]
fig = go.Figure(go.Bar(
    x=net_monthly["year_month"],
    y=net_monthly["net_cashflow"],
    marker_color=colors,
))
fig.add_hline(y=0, line_color="black", line_width=0.8)
fig.update_layout(
    title="Net Cash Flow per Month (excl. transfers)",
    xaxis_title="Month",
    yaxis_title="PLN",
    height=400,
)
fig.show()

## Deposits vs Withdrawals split

Break the net cash flow into its two components and derive disposable income per month.
`type` column values: `deposit` (income, positive), `withdrawal` (expense, sign-flipped to positive),
`transfer` (excluded).
Disposable income = deposits − withdrawals. Savings rate = disposable income / deposits.

In [ ]:
# ── Deposits vs Withdrawals monthly split ─────────────────────────────────────
cashflow_split = build_monthly_cashflow_split(df_featured)
print(f"Months: {len(cashflow_split)}")
display(cashflow_split.round(0))

In [ ]:
# ── Visualise: deposits / withdrawals / disposable income ─────────────────────
x = cashflow_split["year_month"]
avg_dep = cashflow_split["deposits"].mean()
avg_wdw = cashflow_split["withdrawals"].mean()
avg_di = cashflow_split["disposable_income"].mean()
di_colors = ["steelblue" if v >= 0 else "tomato" for v in cashflow_split["disposable_income"]]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=("Monthly Deposits vs Withdrawals",
                    "Monthly Disposable Income (Deposits − Withdrawals)"),
)

fig.add_trace(go.Bar(x=x, y=cashflow_split["deposits"], name="Deposits",
                     marker_color="steelblue", opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(x=x, y=cashflow_split["withdrawals"], name="Withdrawals",
                     marker_color="tomato", opacity=0.8), row=1, col=1)
fig.add_hline(y=avg_dep, line_color="steelblue", line_dash="dash",
              annotation_text=f"Avg deposits {avg_dep:.0f}", row=1, col=1)
fig.add_hline(y=avg_wdw, line_color="tomato", line_dash="dash",
              annotation_text=f"Avg withdrawals {avg_wdw:.0f}", row=1, col=1)

fig.add_trace(go.Bar(x=x, y=cashflow_split["disposable_income"],
                     marker_color=di_colors, opacity=0.85, name="Disposable Income"),
              row=2, col=1)
fig.add_hline(y=0, line_color="black", line_width=0.8, row=2, col=1)
fig.add_hline(y=avg_di, line_color="green", line_dash="dash",
              annotation_text=f"Avg {avg_di:.0f}", row=2, col=1)

fig.update_yaxes(title_text="PLN", row=1, col=1)
fig.update_yaxes(title_text="PLN", row=2, col=1)
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_layout(height=600, barmode="group")
fig.show()

avg_sr = cashflow_split["savings_rate"].mean()
print(f"\nAverage savings rate: {avg_sr:.1%}")
print(f"Months with negative disposable income: "
      f"{(cashflow_split['disposable_income'] < 0).sum()} / {len(cashflow_split)}")

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
writer.save_interim(df_featured, "transactions_featured")
writer.save_interim(monthly, "monthly_aggregates")
writer.save_interim(cashflow_split, "monthly_cashflow_split")
print("Saved:")
print("  data/interim/transactions_featured.parquet")
print("  data/interim/monthly_aggregates.parquet")
print("  data/interim/monthly_cashflow_split.parquet")
